In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
RANDOM_STATE = 42

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
print(train.shape, test.shape)

(891, 12) (418, 11)


In [ ]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
#check missing value
train.isnull().mean().sort_values(ascending = False).head(10)

,0
Cabin,0.771044
Age,0.198653
Embarked,0.002245
PassengerId,0.000000
Name,0.000000
Pclass,0.000000
Survived,0.000000
Sex,0.000000
Parch,0.000000
SibSp,0.000000


In [ ]:
#Target Distribution
train['Survived'].value_counts(normalize = True)

,proportion
Survived,
0,0.616162
1,0.383838


In [ ]:
class FeatureCreator(BaseEstimator, TransformerMixin):
  def fit(self , X,y=None):
    X = X.copy()
    self.ticket_counts_ = X['Ticket'].value_counts(dropna = False)
    return self

  def transform(self, X, y=None):
    X_transformed = X.copy()
    # Correcting the regex pattern by escaping the closing square bracket
    X_transformed['Title'] = X_transformed['Name'].str.extract(r'\s*([^\]]+)\.')
    map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs' # Normalize French titles
    }
    X_transformed['Title'] = X_transformed['Title'].replace(map)
    rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Lady', 'Sir', 'Jonkheer', 'Countess', 'Capt', 'Don', 'Dona']
    X_transformed['Title'] = X_transformed['Title'].where(~X_transformed['Title'].isin(rare_titles), other='Rare')

    X_transformed['FamilySize'] = X_transformed['SibSp'].fillna(0) + X_transformed['Parch'].fillna(0) + 1
    X_transformed['IsAlone'] = (X_transformed['FamilySize'] == 1).astype(int)

    X_transformed['TicketGroupSize'] = X_transformed['Ticket'].map(self.ticket_counts_).fillna(1).astype(int)
    X_transformed['FarePerPerson'] = X_transformed['Fare'] / X_transformed['FamilySize'].replace(0, 1)

    X_transformed['Deck'] = X_transformed['Cabin'].astype(str).str[0]
    # Decks 'A' through 'T' are valid, anything else (like 'n' for NaN) is treated as 'U' (Unknown)
    X_transformed['Deck'] = X_transformed['Deck'].where(X_transformed['Deck'].isin(list('ABCDEFGT')), other='U')
    X_transformed['CabinKnown'] = X_transformed['Cabin'].notna().astype(int)
    X_transformed['AgeMissing'] = X_transformed['Age'].isna().astype(int)

    # Drop original columns that are no longer needed or have been transformed
    X_transformed = X_transformed.drop(['Name', 'SibSp', 'Parch', 'Ticket', 'Cabin', 'Fare'], axis=1)

    return X_transformed

In [ ]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Columns that exist after FeatureCreator
categorical_features = ['Sex', 'Embarked', 'Title', 'Deck']
# Treat Pclass as numeric (ordinal). Binary flags are numeric too.
numeric_features = [
'Age', 'Pclass',
'FamilySize', 'IsAlone', 'TicketGroupSize', 'FarePerPerson',
'CabinKnown', 'AgeMissing'
]


numeric_pipe = Pipeline([
('imputer', SimpleImputer(strategy='median', add_indicator=True)),
('scaler', StandardScaler())
])


categorical_pipe = Pipeline([
('imputer', SimpleImputer(strategy='most_frequent')),
('onehot', OneHotEncoder(handle_unknown='ignore'))
])


preprocess = ColumnTransformer([
('num', numeric_pipe, numeric_features),
('cat', categorical_pipe, categorical_features)
])

In [ ]:
X = train.drop(columns=['Survived'])
y = train['Survived']


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
pipe_lr = Pipeline([
('feat', FeatureCreator()),
('prep', preprocess),
('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
])


acc_lr = cross_val_score(pipe_lr, X, y, cv=cv, scoring='accuracy')
roc_lr = cross_val_score(pipe_lr, X, y, cv=cv, scoring='roc_auc')
print(f"LogReg CV Accuracy: {acc_lr.mean():.4f} ± {acc_lr.std():.4f}")
print(f"LogReg CV ROC AUC: {roc_lr.mean():.4f} ± {roc_lr.std():.4f}")

LogReg CV Accuracy: 0.7980 ± 0.0271
LogReg CV ROC AUC: 0.8569 ± 0.0175


In [ ]:
pipe_rf = Pipeline([
('feat', FeatureCreator()),
('prep', preprocess),
('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'))
])


param_grid_rf = {
'model__n_estimators': [300, 500],
'model__max_depth': [None, 6, 10],
'model__min_samples_split': [2, 5],
'model__min_samples_leaf': [1, 2, 4],
'model__max_features': ['sqrt', 'log2']
}


gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
gs_rf.fit(X, y)
print("Best RF params:", gs_rf.best_params_)
print("Best RF CV ROC AUC:", gs_rf.best_score_)


pipe_gb = Pipeline([
('feat', FeatureCreator()),
('prep', preprocess),
('model', GradientBoostingClassifier(random_state=RANDOM_STATE))
])


param_grid_gb = {
'model__n_estimators': [200, 400],
'model__learning_rate': [0.03, 0.05, 0.1],
'model__max_depth': [2, 3],
'model__subsample': [0.8, 1.0]
}


gs_gb = GridSearchCV(pipe_gb, param_grid_gb, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
gs_gb.fit(X, y)
print("Best GB params:", gs_gb.best_params_)
print("Best GB CV ROC AUC:", gs_gb.best_score_)

Best RF params: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Best RF CV ROC AUC: 0.8801103398896636
Best GB params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 400, 'model__subsample': 1.0}
Best GB CV ROC AUC: 0.8819794646291632


In [ ]:
# Example: choose best of RF vs GB
best_estimator = gs_rf if gs_rf.best_score_ >= gs_gb.best_score_ else gs_gb
best_model = best_estimator.best_estimator_


# Fit on full training data (gs already refit=True)
# Predict on test
X_test = test.copy()
y_pred = best_model.predict(X_test)


submission = pd.DataFrame({
'PassengerId': test['PassengerId'],
'Survived': y_pred.astype(int)
})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')

In [31]:
# Example: choose best of RF vs GB
best_estimator = gs_rf if gs_rf.best_score_ >= gs_gb.best_score_ else gs_gb
best_model = best_estimator.best_estimator_


# Fit on full training data (gs already refit=True)
# Predict on test
X_test = test.copy()
y_pred = best_model.predict(X_test)


submission = pd.DataFrame({
'PassengerId': test['PassengerId'],
'Survived': y_pred.astype(int)
})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')

Saved submission.csv


In [32]:
# Make predictions on the training data for error analysis
y_train_pred = best_model.predict(X)

# Display confusion matrix
print("Confusion Matrix on Training Data:")
display(confusion_matrix(y, y_train_pred))

# Display classification report
print("\nClassification Report on Training Data:")
display(classification_report(y, y_train_pred))

Confusion Matrix on Training Data:


array([[545,   4],
       [ 31, 311]])


Classification Report on Training Data:


'              precision    recall  f1-score   support\n\n           0       0.95      0.99      0.97       549\n           1       0.99      0.91      0.95       342\n\n    accuracy                           0.96       891\n   macro avg       0.97      0.95      0.96       891\nweighted avg       0.96      0.96      0.96       891\n'